In [1]:
import pandas as pd
import numpy as np

from tqdm import tqdm

import matplotlib.pyplot as plt
import plotly.express as px

import utility_functions as uf

In [2]:
year = 2022
mode = "individual"

df_cs = pd.read_csv(uf.PATH+f"df_country_subfield/{mode}_{year}.csv", index_col=0)
df_prob = df_cs.div(df_cs.sum(axis=1), axis=0)

subfield_tree = uf.get_subfield_tree(uf.df_topics)

df_dist = uf.get_w1_distances(subfield_tree, df_prob)
np.fill_diagonal(df_dist.values, np.nan)

In [3]:
start_year = 1970
end_year = 2023

In [ ]:
## calculate distances for each year and save to csv

# df_dist_yearly_list = []
# for year_loop in tqdm(range(start_year, end_year+1)):
#     df_cs = pd.read_csv(uf.PATH+f"df_country_subfield/{mode}_{year_loop}.csv", index_col=0)
#     df_prob = df_cs.div(df_cs.sum(axis=1), axis=0)

#     df_dist = uf.get_w1_distances(subfield_tree, df_prob)
#     np.fill_diagonal(df_dist.values, np.nan)

#     df_dist_yearly_list.append(df_dist.reset_index(names="country").assign(year=year_loop))

# Correlations

In [ ]:
subfield = "1202"
df_prob = pd.read_csv(uf.PATH+f"df_prob_yearly.csv").query("mode == @mode")[["country", "year", subfield]]
df_corr = df_prob.pivot(index="year", columns="country", values=subfield).corr(method="pearson")

In [40]:
uf.get_subfield_info(1202)

,subfield_id,subfield_name,field_name,domain_name
2075,1202,History,Arts and Humanities,Social Sciences


In [41]:
labels = df_cs.sum(axis=1).sort_values(ascending=False).index.to_list()

uf.plotly_heatmap(df_corr.loc[labels, labels], title=f"Correlation matrix of subfield probabilities, {mode} {year}",
                  x_labels=labels, y_labels=labels, line_height=10)


# Mean dist yearly

In [ ]:
# df_dist_yearly = pd.concat(df_dist_yearly_list)
df_dist_yearly = pd.read_csv(uf.PATH+"df_dist_countries_yearly_w1_individual.csv")

df_dist_yearly_flat = (
    df_dist_yearly
    .melt(
        id_vars=["country", "year"], var_name="country_dist", value_name="dist")
    .dropna(subset="dist")
)

In [15]:
country_list = ["US", "FR", "CN", "CH", "IN", "ID", "AU", "BR", "RU"]
fig = px.box(
    (
        df_dist_yearly_flat
        .groupby(["country", "year"], as_index=False)
        .dist
        .mean()
        # .query("year > @test_year")
    ),
    x="year",
    y="dist",
    points="outliers",
    hover_data=["country"],
    title=f"W1-tree Distance between countries ({mode})",

)
fig.update_traces(
    marker=dict(color="grey"),
    line=dict(color="grey"),
    opacity=0.5
)
for country in country_list:
    fig.add_scatter(
        x=df_dist_yearly_flat.query("country == @country").groupby("year", as_index=False).dist.mean().year,
        y=df_dist_yearly_flat.query("country == @country").groupby("year", as_index=False).dist.mean().dist,
        mode="lines",
        name=country,
        # line=dict(width=3)
    )
fig.show()
# fig.write_image("../images/w1_tree_box_1970_2023.png",
#                 width=1200, height=600, scale=2)